# multi_1h_jma_atr_023

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (190).ipynb`

| Field | Value |
|---|---|
| Section | `strategies` |
| Market | `unknown` |
| Trading style | `unknown` |
| Timeframe | `1h` |
| Code cells | 15 |
| Detected functions | resample_1hr_custom, compute_jma_indicator, cy_backtest_trades, backtest_trades, calculate_metrics_fast, __init__, __call__, tqdm_parallel, run_combo, backtest_trades |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** JMA + ATR crossover (detected)
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
data = pd.read_parquet('/content/drive/MyDrive/Data/Binance_Crypto_T7_2020_25.parquet')
data.rename(columns={
    'Datetime': 'Date',
    'Open': 'open',
    'Close': 'close',
    'Low': 'low',
    'High': 'high',
    'Volume': 'volume'
}, inplace=True)

data

In [ ]:
data['Ticker'].unique()

In [ ]:
# Convert 'Date' column to datetime if it isn't already
data['Date'] = pd.to_datetime(data['Date'])

# Define the date range
start_date = '2024-01-01'
end_date = '2024-12-31'

# Define the tickers you want to filter
selected_tickers = ['SOLUSDT']  # Add tickers you want

# Apply filtering
data = data[
    (data["Date"] >= start_date) &
    (data["Date"] <= end_date) &
    (data["Ticker"].isin(selected_tickers))
]

# Display filtered results
data

In [ ]:
import pandas as pd

# Ensure Date is datetime
data['Date'] = pd.to_datetime(data['Date'])

# Custom resampling function with offset
def resample_1hr_custom(group):
    ticker = group['Ticker'].iloc[0]
    group = group.set_index('Date')
    resampled = group.resample('1h').agg({
        'open': 'first',
        'high': 'max',
        'low': 'min',
        'close': 'last',
        'volume': 'sum'
    }).dropna().reset_index()
    resampled['Ticker'] = ticker
    return resampled

# Apply per ticker
data_1hr = data.groupby('Ticker').apply(resample_1hr_custom).reset_index(drop=True)

# Reorder columns
data_1hr = data_1hr[['Ticker', 'Date', 'open', 'high', 'low', 'close', 'volume']]

data = data_1hr
data

In [ ]:
data.dropna()
data

In [ ]:
data.columns

In [ ]:
# --- PART 1: Setup + JMA Function (optimized) ---

import pandas as pd
import numpy as np
from tqdm import tqdm
from joblib import Parallel, delayed

def compute_jma_indicator(
    df,
    smooth_length=14,
    normalization_period=96,
    upper_threshold=1,
    lower_threshold=-2,
    rs_period=21,
    rs_percentile=30
):
    price_series = df['close'].values
    alpha = 2.0 / (smooth_length + 1)
    jma_values = np.empty_like(price_series)
    jma_values[0] = price_series[0]

    for i in range(1, len(price_series)):
        jma_values[i] = jma_values[i-1] + (price_series[i] - jma_values[i-1]) * alpha

    jma_series = pd.Series(jma_values, index=df.index)

    rolling_mean = jma_series.rolling(window=normalization_period).mean()
    rolling_std = jma_series.rolling(window=normalization_period).std()
    normalized_jma = (jma_series - rolling_mean) / rolling_std

    open_ = df['open'].replace(0, np.nan)
    high = df['high']
    low = df['low']
    close = df['close'].replace(0, np.nan)

    log_hc = np.log(high / close)
    log_ho = np.log(high / open_)
    log_lc = np.log(low / close)
    log_lo = np.log(low / open_)

    rs_volatility = (log_hc * log_ho + log_lc * log_lo).rolling(rs_period).mean()
    rs_threshold = rs_volatility.rolling(window=normalization_period).quantile(rs_percentile / 100.0)

    long_signals = (normalized_jma > upper_threshold) & (rs_volatility > rs_threshold)
    short_signals = (normalized_jma < lower_threshold) & (rs_volatility > rs_threshold)

    signals = pd.Series(0, index=df.index)
    signals[long_signals] = 1
    signals[short_signals] = 2

    df['JMA'] = jma_series
    df['Normalized_JMA'] = normalized_jma
    df['RS_Volatility'] = rs_volatility
    df['RS_Threshold'] = rs_threshold
    df['Signal'] = signals

    return df


In [ ]:
%load_ext cython

In [ ]:
# --- PART 2: Backtester + Metric Functions ---

# Run this only ONCE at the top of your Colab

%%cython
import numpy as np
cimport numpy as np

def cy_backtest_trades(
    object[:, :] data,
    str ticker,
    double capital,
    double fee_rate,
    double slippage_rate,
    double tp_pct,
    double sl_pct,
    double entry_sl_pct
):
    cdef:
        int i, n = data.shape[0]
        bint position_open = False
        bint exit_next_open = False

        double entry_price = 0.0
        object entry_time = None
        int direction = 0
        double quantity = 0.0
        double cumulative_pnl = 0.0
        object last_exit_time = None
        list trades = []

        double tp_price = 0.0, sl_price = 0.0, esl_price = 0.0
        double exit_price = 0.0
        double open_, high, low, close
        double pnl = 0.0, fee = 0.0, net_pnl = 0.0
        object time
        int signal
        str reason

    for i in range(n):
        time = data[i, 0]
        open_ = <double>data[i, 1]
        high = <double>data[i, 2]
        low = <double>data[i, 3]
        close = <double>data[i, 4]
        signal = <int>data[i, 5]

        if exit_next_open:
            exit_price = open_ * (1 + slippage_rate) if direction == -1 else open_ * (1 - slippage_rate)
            pnl = (exit_price - entry_price) * quantity * direction
            fee = fee_rate * (entry_price + exit_price) * quantity
            net_pnl = pnl - fee
            cumulative_pnl += net_pnl
            trades.append((ticker, entry_time, time, entry_price, exit_price, net_pnl,
                           "SL Close Confirmed", cumulative_pnl, "long" if direction == 1 else "short"))
            position_open = False
            exit_next_open = False
            last_exit_time = time
            continue

        if not position_open and time != last_exit_time:
            if signal not in (1, 2):
                continue
            direction = 1 if signal == 1 else -1
            entry_price = open_ * (1 + slippage_rate) if direction == 1 else open_ * (1 - slippage_rate)
            quantity = capital / entry_price
            entry_time = time
            position_open = True
            tp_price = entry_price * (1 + tp_pct * direction)
            sl_price = entry_price * (1 - sl_pct * direction)
            esl_price = entry_price * (1 - entry_sl_pct * direction)

            if (direction == 1 and low <= esl_price) or (direction == -1 and high >= esl_price):
                exit_price = esl_price
                pnl = (exit_price - entry_price) * quantity * direction
                fee = fee_rate * (entry_price + exit_price) * quantity
                net_pnl = pnl - fee
                cumulative_pnl += net_pnl
                trades.append((ticker, entry_time, time, entry_price, exit_price, net_pnl,
                               "Entry SL", cumulative_pnl, "long" if direction == 1 else "short"))
                position_open = False
                last_exit_time = time
                continue

            if (direction == 1 and close <= sl_price) or (direction == -1 and close >= sl_price):
                exit_next_open = True
            continue

        if position_open:
            if direction == 1:
                new_sl = high * (1 - sl_pct)
                if new_sl > sl_price:
                    sl_price = new_sl
            else:
                new_sl = low * (1 + sl_pct)
                if new_sl < sl_price:
                    sl_price = new_sl

            if (direction == 1 and high >= tp_price):
                exit_price = tp_price
                reason = "Take Profit"
            elif (direction == -1 and low <= tp_price):
                exit_price = tp_price
                reason = "Take Profit"
            elif (direction == 1 and low <= sl_price):
                exit_price = sl_price
                reason = "Stop Loss"
            elif (direction == -1 and high >= sl_price):
                exit_price = sl_price
                reason = "Stop Loss"
            else:
                continue

            pnl = (exit_price - entry_price) * quantity * direction
            fee = fee_rate * (entry_price + exit_price) * quantity
            net_pnl = pnl - fee
            cumulative_pnl += net_pnl
            trades.append((ticker, entry_time, time, entry_price, exit_price, net_pnl,
                           reason, cumulative_pnl, "long" if direction == 1 else "short"))
            position_open = False
            last_exit_time = time

    return trades

# Wrapper
def backtest_trades(data, tp_pct, sl_pct, entry_sl_pct):
    data['Date'] = pd.to_datetime(data['Date'])
    data['Date'] = np.array(data['Date'].dt.to_pydatetime())
    tickers = data['Ticker'].unique()

    ticker_map = {
        t: data[data['Ticker'] == t][["Date", "open", "high", "low", "close", "Signal"]].to_numpy(object)
        for t in tickers
    }

    all_trades = []
    for ticker in tickers:
        trades = cy_backtest_trades(
            ticker_map[ticker], ticker,
            100000, 0.0004, 0.0005,
            tp_pct, sl_pct, entry_sl_pct
        )
        all_trades.extend(trades)

    return pd.DataFrame(all_trades, columns=[
        "Ticker", "Entry Time", "Exit Time", "Entry Price", "Exit Price",
        "PnL", "Exit Reason", "Cumulative PnL", "Direction"
    ])

def calculate_metrics_fast(tradebook):
    if tradebook.empty: return 0.0, 0.0, 0.0, 0.0, 0.0, 0
    pnls = tradebook['PnL'].values
    equity = np.cumsum(pnls)
    roll_max = np.maximum.accumulate(equity)
    max_dd = np.max(roll_max - equity)
    ratio = equity[-1] / max_dd if max_dd > 0 else 0.0
    x = np.arange(len(equity))
    slope = np.dot(x - x.mean(), equity - equity.mean()) / np.dot(x - x.mean(), x - x.mean())
    y_pred = slope * (x - x.mean()) + equity.mean()
    r2 = 1 - np.sum((equity - y_pred)**2) / np.sum((equity - equity.mean())**2)
    months = pd.to_datetime(tradebook['Entry Time']).values.astype('datetime64[M]')
    monthly_pnl = pd.Series(pnls).groupby(months).sum().values
    consistency = monthly_pnl.mean() / monthly_pnl.std() if len(monthly_pnl) > 1 else 0.0
    return ratio, equity[-1], r2, consistency, np.mean(pnls > 0), len(pnls)


In [ ]:
# --- PART 3: Hyperparameter Grid + Parallel Runner ---

from itertools import product
import os, pickle

# Grid
param_grid = {
    'smooth_length': [10, 14, 21],
    'norm_period': [84, 96, 108],
    'upper_threshold': [0.75, 1, 1.25, 1.5, 1.75],
    'lower_threshold': [-1.25, -1.5, -1.75, -2, -2.25],
    'rs_period': [3, 10, 14, 21, 28, 32],
    'rs_percentile': [30, 50, 70],
    'tp_pct': [0.04, 0.05, 0.06],
    'sl_pct': [0.001, 0.002, 0.003],
    'entry_sl_pct': [0.02, 0.03, 0.04]
}
keys, values = zip(*param_grid.items())
param_combinations = list(product(*values))

# Output
csv_path = "/content/drive/MyDrive/optimized_results.csv"
cache_path = "/content/drive/MyDrive/optimized_checkpoint.pkl"

# Resume
start_index = 0
results = []
if os.path.exists(cache_path):
    with open(cache_path, 'rb') as f:
        checkpoint = pickle.load(f)
        start_index = checkpoint['last_index'] + 1
        results = checkpoint['results']


In [ ]:
import pandas as pd
import pickle
from tqdm import tqdm
from joblib import Parallel, delayed
from joblib.parallel import BatchCompletionCallBack

# === Monkey-patch for tqdm with joblib ===
class TqdmBatchCompletionCallBack(BatchCompletionCallBack):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        if not hasattr(tqdm, '_tqdm_bar'):
            tqdm._tqdm_bar = tqdm(total=self.parallel.n_dispatched_batches * self.parallel.batch_size)

    def __call__(self, *args, **kwargs):
        tqdm._tqdm_bar.update(self.batch_size)
        return super().__call__(*args, **kwargs)

Parallel.__call__old = Parallel.__call__
def tqdm_parallel(self, *args, **kwargs):
    self._backend.BatchCompletionCallBack = TqdmBatchCompletionCallBack
    return self.__call__old(*args, **kwargs)
Parallel.__call__ = tqdm_parallel

# === Tuning Execution Function ===
def run_combo(idx, combo):
    param = dict(zip(keys, combo))
    try:
        temp_data = data.copy()
        temp_data = temp_data.groupby("Ticker", group_keys=False).apply(
            lambda df: compute_jma_indicator(
                df,
                smooth_length=param['smooth_length'],
                normalization_period=param['norm_period'],
                upper_threshold=param['upper_threshold'],
                lower_threshold=param['lower_threshold'],
                rs_period=param['rs_period'],
                rs_percentile=param['rs_percentile']
            )
        )

        temp_data['Signal'] = temp_data.groupby('Ticker')['Signal'].shift(1)
        temp_data.dropna(subset=['Signal'], inplace=True)
        temp_data['Signal'] = temp_data['Signal'].astype(int)

        tradebook = backtest_trades(
            temp_data,
            tp_pct=param['tp_pct'],
            sl_pct=param['sl_pct'],
            entry_sl_pct=param['entry_sl_pct']
        )

        metrics = calculate_metrics_fast(tradebook)
        result = {
            **param,
            'Profit/Drawdown': metrics[0],
            'Total Profit': metrics[1],
            'Equity R2': metrics[2],
            'Monthly Consistency': metrics[3],
            'Win Rate': metrics[4],
            'Total Trades': metrics[5],
        }
        result['Composite Score'] = (
            result['Profit/Drawdown'] * 0.3 +
            result['Equity R2'] * 0.5 +
            result['Monthly Consistency'] * 0.2
        )
        return idx + start_index, result
    except Exception as e:
        print(f"❌ Error at combo {idx + start_index}: {param} — {e}")
        return idx + start_index, None

# === Parallel Execution ===
SAVE_EVERY = 50
param_gen = list(enumerate(param_combinations[start_index:]))

results_batch = Parallel(n_jobs=-1)(
    delayed(run_combo)(i, combo) for i, combo in param_gen
)

# === Save Results with Checkpointing ===
for idx, result in results_batch:
    if result:
        results.append(result)
    if len(results) % SAVE_EVERY == 0 or idx == len(param_combinations) - 1:
        df = pd.DataFrame(results)
        df.to_csv(csv_path, index=False)
        with open(cache_path, 'wb') as f:
            pickle.dump({'last_index': idx, 'results': results}, f)
        print(f"💾 Saved checkpoint at index {idx}")


In [ ]:
from tqdm import tqdm
import numpy as np
import pandas as pd

def backtest_trades(
    data: pd.DataFrame,
    tp_pct: float,
    sl_pct: float,
    entry_sl_pct: float,
) -> pd.DataFrame:

    initial_capital = 100000
    fee_rate = 0.0004  # 0.04% round trip
    slippage_rate = 0.0005  # 0.05% round trip

    # Convert 'Date' to datetime and avoid deprecated warning
    data['Date'] = pd.to_datetime(data['Date'])
    data['Date'] = np.array(data['Date'].dt.to_pydatetime())

    tickers = data['Ticker'].unique()

    # Prepare each ticker's data for cython backtest
    ticker_map = {
        ticker: data[data['Ticker'] == ticker][[
            "Date", "open", "high", "low", "close", "Signal"
        ]].to_numpy(dtype=object)
        for ticker in tickers
    }

    all_trades = []

    for ticker in tqdm(tickers, desc="Backtesting"):
        trades = cy_backtest_trades(
            ticker_map[ticker],
            ticker,
            initial_capital,
            fee_rate,
            slippage_rate,
            tp_pct,
            sl_pct,
            entry_sl_pct  # Used for Entry SL check
        )
        all_trades.extend(trades)

    return pd.DataFrame(all_trades, columns=[
        "Ticker", "Entry Time", "Exit Time", "Entry Price", "Exit Price",
        "PnL", "Exit Reason", "Cumulative PnL", "Direction"
    ])

tradebook = backtest_trades(
    data,
    tp_pct=0.05,            # 0.5% TP
    sl_pct=0.001,            # 0.2% SL
    entry_sl_pct=0.03,      # 0.3% Entry SL
)

tradebook.to_csv("/content/drive/MyDrive/Data/tradebook_SOLUSDT_2022_24.csv", index=False)
tradebook
